# Imports

In [ ]:
import sys
import pandas as pd
from pathlib import Path
import plotnine
from plotnine import *
from mizani.formatters import comma_format
import patchworklib as pwl

repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from experiments.utils import load_experiments_specs, auto_layout_plots  # noqa: E402

results_dir = repo_root / 'experiments' / 'runs' / 'results'
plots_dir = results_dir / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)

plotnine.options.figure_size = (18, 14)

# Load Forecasts

In [ ]:
datasets_config = [
    {'dataset': 'auselectricity',  'display': 'Australian Electricity Demand', 'fcst_h': 24, 'sample_ids': ['T1', 'T2', 'T5']},
    {'dataset': 'ausretail',       'display': 'Australian Retail',             'fcst_h': 24, 'sample_ids': ['A3349609R', 'A3349643V', 'A3349483V']},
    {'dataset': 'm3_monthly',      'display': 'M3 Monthly',                    'fcst_h': 18, 'sample_ids': ['T102', 'T1068', 'T1170']},
    {'dataset': 'm3_yearly',       'display': 'M3 Yearly',                     'fcst_h': 6,  'sample_ids': ['Y97', 'Y39', 'Y147']},
    {'dataset': 'm5_agg',          'display': 'M5',                            'fcst_h': 28, 'sample_ids': ['FOODS_1_FOODS_WI_2_WI', 'HOBBIES_1_HOBBIES_WI_1_WI', 'HOBBIES_2_HOBBIES_CA_1_CA']},
    {'dataset': 'rossmann',        'display': 'Rossmann Store Sales',          'fcst_h': 40, 'sample_ids': ['1015', '743', '86']},
    {'dataset': 'tourism_monthly', 'display': 'Tourism Monthly',               'fcst_h': 24, 'sample_ids': ['T26', 'T32', 'T33']},
]

frames = []
for cfg in datasets_config:
    dataset = cfg['dataset']
    display_name = cfg['display']
    fcst_h = cfg['fcst_h']
    sample_ids = [str(s) for s in cfg['sample_ids']]

    fcst_path = results_dir / 'global' / f'{dataset}_hypertrees_fcsts.csv'
    if not fcst_path.exists():
        print(f'[skipped] {dataset}: no forecast CSV')
        continue

    fcst = pd.read_csv(fcst_path, parse_dates=['date'])
    fcst['series_id'] = fcst['series_id'].astype(str)
    fcst['model'] = fcst['model'].str.replace(r'\(.*?\)', '', regex=True)
    fcst = fcst.dropna(subset=['fcst'])
    fcst = fcst[fcst['series_id'].isin(sample_ids)]
    fcst_rows = fcst[['series_id', 'date', 'fcst', 'model']].copy()

    specs = load_experiments_specs(dataset=dataset, train_type='global')
    train_all = specs['train'][['series_id', 'date', 'value']].copy()
    train_all['series_id'] = train_all['series_id'].astype(str)
    test_all = specs['test'][['series_id', 'date', 'value']].copy()
    test_all['series_id'] = test_all['series_id'].astype(str)

    train_actuals = train_all[train_all['series_id'].isin(sample_ids)]
    test_actuals = test_all[test_all['series_id'].isin(sample_ids)]
    actual_rows = (
        pd.concat([train_actuals, test_actuals], axis=0, ignore_index=True)
        .sort_values(['series_id', 'date'])
        .rename(columns={'value': 'fcst'})
    )
    actual_rows['model'] = 'Actual'

    ds_df = pd.concat([fcst_rows, actual_rows], axis=0, ignore_index=True)
    ds_df['series_id_plot'] = ds_df['series_id'].apply(lambda s: f'{display_name}: {s}')
    ds_df['fcst_h'] = fcst_h
    frames.append(ds_df)

fcsts_df = pd.concat(frames, axis=0, ignore_index=True)
fcsts_df['date'] = pd.to_datetime(fcsts_df['date'])

# Preserve (dataset, sample-id) order from the config for the flat `unique_ids` list.
unique_ids = [
    f"{cfg['display']}: {sid}"
    for cfg in datasets_config
    for sid in [str(s) for s in cfg['sample_ids']]
    if f"{cfg['display']}: {sid}" in fcsts_df['series_id_plot'].unique()
]

fcsts_df.head()

# Plots

In [ ]:
fcst_plots = []
for series_sub in unique_ids:

    ###
    # Create Dataset
    ###
    data_sub = fcsts_df[fcsts_df["series_id_plot"] == series_sub].reset_index(drop=True).copy()
    fcst_h = data_sub["fcst_h"].values[0]
    ctx_len = 4*fcst_h
    test_df = data_sub[data_sub["model"] == "Actual"].tail(fcst_h)
    fcst_df_sub = fcsts_df[fcsts_df["series_id_plot"] == series_sub].reset_index(drop=True)

    ###
    # HyperTree-AR
    ###
    hyper_tree_ar_fcst = fcst_df_sub[fcst_df_sub["model"] == "Hyper-Tree-AR"]

    ###
    # HyperTree-ES
    ###
    hyper_tree_es_fcst = fcst_df_sub[fcst_df_sub["model"] == "Hyper-Tree-ETS"]

    ###
    # GBMNet-AR
    ###
    hyper_treenet_ar_fcst = fcst_df_sub[fcst_df_sub["model"] == "Hyper-TreeNet-AR"]

    ###
    # Actual
    ###
    actual_df = fcst_df_sub[fcst_df_sub["model"] == f"Actual"].iloc[-ctx_len:,:]

    # Fcst
    fcst_df = pd.concat(
        [
            actual_df,
            hyper_tree_ar_fcst,
            hyper_tree_es_fcst,
            hyper_treenet_ar_fcst

        ],
        axis=0
    )
    fcst_df["date"] = pd.to_datetime(fcst_df["date"])


    if "Hyper-Tree-ETS" in fcst_df["model"].unique():
        fcst_df["model"] = pd.Categorical(
            fcst_df["model"],
            categories=["Actual",
                        "Hyper-Tree-AR",
                        "Hyper-Tree-ETS",
                        "Hyper-TreeNet-AR"]
        )
        color_map = {
            "Actual": "black",
            "Hyper-Tree-AR": "#d62728",
            "Hyper-Tree-ETS": "#FFA500",
            "Hyper-TreeNet-AR": "#2ca02c"
        }
        size_map = {
            "Actual": 0.7,
            "Hyper-Tree-AR": 1.4,
            "Hyper-Tree-ETS": 1.4,
            "Hyper-TreeNet-AR": 1.4
        }
        line_map = {"Actual": "solid",
                     "Hyper-Tree-AR": "solid",
                     "Hyper-Tree-ETS": "solid",
                     "Hyper-TreeNet-AR": "solid",
                     }
    else:
        fcst_df["model"] = pd.Categorical(
            fcst_df["model"],
            categories=["Actual",
                        "Hyper-Tree-AR",
                        "Hyper-TreeNet-AR"]
        )
        color_map = {
            "Actual": "black",
            "Hyper-Tree-AR": "#d62728",
            "Hyper-TreeNet-AR": "#2ca02c"
        }
        size_map = {
            "Actual": 1.5,
            "Hyper-Tree-AR": 2.5,
            "Hyper-TreeNet-AR": 2.5
        }
        line_map = {"Actual": "solid",
                    "Hyper-Tree-AR": "solid",
                    "Hyper-TreeNet-AR": "solid",
                    }

    fcst_plots.append(
        pwl.load_ggplot(
            ggplot(fcst_df,
                   aes(x="date",
                       y="fcst",
                       color="model")) +
            geom_line(aes(size='model', linetype='model')) +
            facet_wrap("series_id_plot") +
            geom_vline(xintercept=test_df["date"].min(), linetype="dashed") +
            scale_color_manual(values=color_map) +
            scale_size_manual(values=size_map) +
            scale_linetype_manual(values=line_map) +
            theme_bw(base_size=30) +
            theme(legend_position="bottom",
                  legend_title=element_blank(),
                  legend_key=element_rect(fill="none", colour="none"),
                  panel_grid_major_x=element_line(color="grey", alpha=0.05),
                  panel_grid_minor_x=element_line(color="grey", alpha=0.05),
                  panel_grid_major_y=element_line(color="grey", alpha=0.05),
                  panel_grid_minor_y=element_line(color="grey", alpha=0.05)
                  ) +
            scale_y_continuous(labels=comma_format()) +
            labs(title="",
                 x="",
                 y="")
        )
    )

# Combine Plots

In [ ]:
combined_plot1 = auto_layout_plots(fcst_plots[0:12])
combined_plot1.savefig(str(plots_dir / 'example_forecasts1.pdf'), format="pdf", dpi=2000, bbox_inches="tight")
combined_plot1.savefig(str(plots_dir / 'example_forecasts1.png'), format="png", dpi=150, bbox_inches="tight")

combined_plot2 = auto_layout_plots(fcst_plots[12:])
combined_plot2.savefig(str(plots_dir / 'example_forecasts2.pdf'), format="pdf", dpi=2000, bbox_inches="tight")
combined_plot2.savefig(str(plots_dir / 'example_forecasts2.png'), format="png", dpi=150, bbox_inches="tight")